# 4 — Dynamic INT8 quantization
Runs in the **Jupyter container on node-serve-model**.
Uses Intel Neural Compressor `approach="dynamic"`. No calibration data needed.

In [ ]:
import time
import numpy as np
import onnxruntime as ort
from neural_compressor import PostTrainingQuantConfig, quantization

ONNX_PATH    = "model_artifacts/smartqueue_ranker.onnx"
DYNAMIC_PATH = "model_artifacts/smartqueue_ranker_dynamic_q.onnx"

def benchmark_session(session, input_dim=64, num_trials=500, batch_size=32):
    name = session.get_inputs()[0].name
    single = np.random.rand(1, input_dim).astype(np.float32)
    for _ in range(20):
        session.run(None, {name: single})
    latencies = []
    for _ in range(num_trials):
        t0 = time.time()
        session.run(None, {name: single})
        latencies.append(time.time() - t0)
    print(f"p50: {np.percentile(latencies, 50)*1000:.2f} ms")
    print(f"p95: {np.percentile(latencies, 95)*1000:.2f} ms")
    print(f"p99: {np.percentile(latencies, 99)*1000:.2f} ms")
    batch = np.random.rand(batch_size, input_dim).astype(np.float32)
    times = []
    for _ in range(200):
        t0 = time.time()
        session.run(None, {name: batch})
        times.append(time.time() - t0)
    print(f"Throughput (batch={batch_size}): {batch_size * 200 / sum(times):.1f} samples/sec")

In [ ]:
config = PostTrainingQuantConfig(approach="dynamic")
q_model = quantization.fit(ONNX_PATH, config)
q_model.save(DYNAMIC_PATH)
print(f"Saved → {DYNAMIC_PATH}")

In [ ]:
print("=== FP32 baseline ===")
benchmark_session(ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"]))

print("\n=== Dynamic INT8 ===")
benchmark_session(ort.InferenceSession(DYNAMIC_PATH, providers=["CPUExecutionProvider"]))